In [12]:
import numpy as np
import pandas as pd
import re

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [13]:
df = pd.read_csv('twitter_dataset.csv')

df.head()

,Tweet_ID,Username,Text,Retweets,Likes,Timestamp
0,1,julie81,Party least receive say or single. Prevent pre...,2,25,2023-01-30 11:00:51
1,2,richardhester,Hotel still Congress may member staff. Media d...,35,29,2023-01-02 22:45:58
2,3,williamsjoseph,Nice be her debate industry that year. Film wh...,51,25,2023-01-18 11:25:19
3,4,danielsmary,Laugh explain situation career occur serious. ...,37,18,2023-04-10 22:06:29
4,5,carlwarren,Involve sense former often approach government...,27,80,2023-01-24 07:12:21


In [14]:
#preprocess
stop_words = set(ENGLISH_STOP_WORDS)

def preprocess(text):
    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+|@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    filtered_words = []

    for word in words:
        if word not in stop_words:
            filtered_words.append(word)

    return filtered_words


tweets = []
original_tweets = []

for text in df['Text'].dropna():
    cleaned = preprocess(text)

    if len(cleaned) > 0:
        tweets.append(cleaned)
        original_tweets.append(text)



In [19]:
model = Word2Vec(
    sentences=tweets,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    epochs=20
)

In [20]:
def get_vector(words):
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


tweets_vec = []

for tweet in tweets:
    tweets_vec.append(get_vector(tweet))  #revise

tweets_vec = np.array(tweets_vec)    

In [22]:
user_input = input("\nEnter user input")
user_tokens = preprocess(user_input)
user_vector = get_vector(user_tokens)


Enter user input party


In [23]:
known_words = False

for word in user_tokens:
    if word in model.wv:
        known_words = True
        break

if known_words == False:
    print("\nNo related info found. Enter some other text.")
else:
    similarities = cosine_similarity(
        user_vector.reshape(1,-1),
        tweets_vec
    )[0]

    top_indices = np.argsort(similarities)[::-1][:5]

    for i in range(5):
        idx = top_indices[i]

        print("Recommendation", i+1)
        print("Tweet: ", original_tweets[idx])
        print("Similarity: ", round(similarities[idx], 4))
        print("--------------")

    

Recommendation 1
Tweet:  Story party official if smile however pull second. Grow the question she.
Town better husband heart religious she. Sometimes those third marriage turn.
Similarity:  0.5205
--------------
Recommendation 2
Tweet:  In ask mean foreign pass. Wrong sing thousand one. Religious pass party always early.
Learn offer off parent everyone. Art street law. Respond discussion answer else cultural success.
Similarity:  0.5191
--------------
Recommendation 3
Tweet:  Executive executive everyone tonight. Material him crime life property five great.
Too sing but once. Fill camera close smile future party or. Mission third process federal today.
Similarity:  0.5165
--------------
Recommendation 4
Tweet:  Want leave still. Husband go result party soon whether season so. Former value Mrs.
Network as within situation worker. Respond kid south response.
Similarity:  0.5116
--------------
Recommendation 5
Tweet:  Positive new wide meeting. Foot always foreign mission. Business arrive